# Mirage: Exposing the Forgetting Illusion in VFL Unlearning
## Representation-Level Auditing Framework — ECCV 2026

This notebook implements the complete experimental pipeline for the paper:
**"Mirage: Exposing the Forgetting Illusion in Vertical Federated Unlearning via Representation-Level Auditing"**

### Contents
1. Setup & Dependencies
2. Configuration & Hyperparameters
3. Data Download & Preprocessing
4. Model Definitions (VFL Architecture)
5. VFL Training Pipeline
6. Unlearning Methods Implementation
7. Standard Evaluation Metrics
8. **Mirage Auditing Framework** (Core Contribution)
9. Experiment Runner
10. Run All Experiments
11. Ablation Studies
12. Visualization & Figure Generation
13. Results Summary

## 1. Setup & Dependencies

In [ ]:
# Install required packages (uncomment if running on Colab)
# !pip install torch torchvision scikit-learn matplotlib seaborn pandas numpy tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset, random_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import os, random, copy, time, warnings, gc
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Configuration & Hyperparameters

In [ ]:
# ============================================================
# Reproducibility
# ============================================================
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ============================================================
# Hyperparameters (matching baseline paper)
# ============================================================
CONFIG = {
    # Training
    'lr': 0.01,
    'batch_size': 128,
    'train_epochs': 50,       # Use 200 for full training
    'weight_decay': 5e-4,
    'momentum': 0.9,
    
    # Unlearning
    'unlearn_epochs': 5,
    'unlearn_lr': 0.01,
    'mixup_alpha': 1.0,       # Beta(1,1) for lambda
    'num_public_samples': 40, # per label
    
    # VFL
    'num_passive_parties': 2,
    
    # Mirage
    'probe_C': 1.0,           # Logistic regression regularization
    'probe_max_iter': 1000,
    'cka_num_samples': 5000,
    'num_seeds': 3,
    
    # Evaluation
    'mia_shadow_epochs': 20,
}

# Output directories
os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)
print("Configuration loaded successfully.")

## 3. Data Download & Preprocessing

We use the same seven datasets as the baseline paper:
- **MNIST**, **CIFAR-10**, **CIFAR-100** (torchvision)
- **ModelNet**, **Brain Tumor MRI**, **COVID-19 Radiography** (download)
- **Yahoo Answers** (text)

For VFL, features are split into K=2 equal partitions along the width dimension.

In [ ]:
# ============================================================
# VFL Feature Splitting Dataset Wrapper
# ============================================================
class VFLDataset(Dataset):
    """Wraps a dataset to split features for VFL parties."""
    def __init__(self, dataset, num_parties=2):
        self.dataset = dataset
        self.num_parties = num_parties

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        if x.dim() == 3:  # C x H x W — split along width
            w = x.shape[2]
            chunk = w // self.num_parties
            parts = []
            for k in range(self.num_parties):
                s = k * chunk
                e = s + chunk if k < self.num_parties - 1 else w
                parts.append(x[:, :, s:e])
            return parts, y
        else:  # flat features — split along feature dim
            d = x.shape[0]
            chunk = d // self.num_parties
            parts = []
            for k in range(self.num_parties):
                s = k * chunk
                e = s + chunk if k < self.num_parties - 1 else d
                parts.append(x[s:e])
            return parts, y


# ============================================================
# Custom Dataset: ModelNet10 (.off -> 32x32 depth image)
# ============================================================
class ModelNet10Dataset(Dataset):
    def __init__(self, root, train=True, transform=None, img_size=32):
        self.transform = transform; self.img_size = img_size
        split = 'train' if train else 'test'
        base = os.path.join(root, 'ModelNet10')
        self.classes = sorted([d for d in os.listdir(base)
                               if os.path.isdir(os.path.join(base, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples, self.targets = [], []
        for cls in self.classes:
            cd = os.path.join(base, cls, split)
            if not os.path.isdir(cd): continue
            for f in sorted(os.listdir(cd)):
                if f.endswith('.off'):
                    self.samples.append(os.path.join(cd, f))
                    self.targets.append(self.class_to_idx[cls])

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img = self._render(self.samples[idx])
        if self.transform: img = self.transform(img)
        return img, self.targets[idx]

    def _render(self, path):
        verts = self._read_off(path); sz = self.img_size
        img = np.zeros((sz, sz), dtype=np.float32)
        if len(verts) == 0: return torch.FloatTensor(img).unsqueeze(0)
        verts = verts - verts.mean(0); mx = np.abs(verts).max()
        if mx > 1e-6: verts /= mx
        xs = np.clip(((verts[:,0]+1)/2*(sz-1)).astype(int), 0, sz-1)
        ys = np.clip(((verts[:,1]+1)/2*(sz-1)).astype(int), 0, sz-1)
        zs = (verts[:,2]+1)/2
        for x,y,z in zip(xs,ys,zs): img[y,x] = max(img[y,x], z)
        return torch.FloatTensor(img).unsqueeze(0)

    @staticmethod
    def _read_off(path):
        with open(path,'r') as f:
            ln = f.readline().strip()
            if ln == 'OFF': parts = f.readline().strip().split()
            elif ln.startswith('OFF'): parts = ln[3:].strip().split()
            else: parts = ln.split()
            nv = int(parts[0]); verts = []
            for _ in range(nv):
                vs = f.readline().strip().split()
                if len(vs)>=3: verts.append([float(vs[0]),float(vs[1]),float(vs[2])])
        return np.array(verts) if verts else np.zeros((0,3))


# ============================================================
# Custom Dataset: ImageFolder-style for BrainTumor / COVID19
# ============================================================
from PIL import Image as PILImage
from glob import glob as _glob

class ImageFolderFlat(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root)
                               if os.path.isdir(os.path.join(root, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        exts = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
        self.samples, self.targets = [], []
        for cls in self.classes:
            cd = os.path.join(root, cls)
            for f in sorted(os.listdir(cd)):
                if os.path.splitext(f)[1].lower() in exts:
                    self.samples.append(os.path.join(cd, f))
                    self.targets.append(self.class_to_idx[cls])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img = PILImage.open(self.samples[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.targets[idx]


# ============================================================
# Custom Dataset: TensorDataset with .targets
# ============================================================
class TensorDatasetWithTargets(Dataset):
    def __init__(self, features, labels):
        self.features = features if isinstance(features, torch.Tensor) else torch.FloatTensor(features)
        self.labels = labels if isinstance(labels, torch.Tensor) else torch.LongTensor(labels)
        self.targets = self.labels.tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.features[idx], self.labels[idx]


# ============================================================
# Dataset Loading — all 7 datasets
# ============================================================
def get_dataset(name='CIFAR10', data_root='./data_raw'):
    """Returns (train_ds, test_ds, num_classes, input_channels, img_size).
    For tabular (YahooAnswers): input_channels=0, img_size=feature_dim.
    """
    if name == 'MNIST':
        t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,),(0.3081,))])
        train_ds = torchvision.datasets.MNIST(data_root, True, download=True, transform=t)
        test_ds  = torchvision.datasets.MNIST(data_root, False, download=True, transform=t)
        return train_ds, test_ds, 10, 1, 28

    elif name == 'CIFAR10':
        tr = transforms.Compose([transforms.RandomCrop(32,4), transforms.RandomHorizontalFlip(),
             transforms.ToTensor(), transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))])
        te = transforms.Compose([transforms.ToTensor(),
             transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))])
        train_ds = torchvision.datasets.CIFAR10(data_root, True, download=True, transform=tr)
        test_ds  = torchvision.datasets.CIFAR10(data_root, False, download=True, transform=te)
        return train_ds, test_ds, 10, 3, 32

    elif name == 'CIFAR100':
        tr = transforms.Compose([transforms.RandomCrop(32,4), transforms.RandomHorizontalFlip(),
             transforms.ToTensor(), transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761))])
        te = transforms.Compose([transforms.ToTensor(),
             transforms.Normalize((0.5071,0.4867,0.4408),(0.2675,0.2565,0.2761))])
        train_ds = torchvision.datasets.CIFAR100(data_root, True, download=True, transform=tr)
        test_ds  = torchvision.datasets.CIFAR100(data_root, False, download=True, transform=te)
        return train_ds, test_ds, 100, 3, 32

    elif name == 'ModelNet10':
        t = transforms.Normalize((0.5,),(0.5,))
        train_ds = ModelNet10Dataset(data_root, True,  transform=t, img_size=32)
        test_ds  = ModelNet10Dataset(data_root, False, transform=t, img_size=32)
        return train_ds, test_ds, len(train_ds.classes), 1, 32

    elif name == 'BrainTumor':
        tr = transforms.Compose([transforms.Resize((32,32)), transforms.RandomHorizontalFlip(),
             transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
        te = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor(),
             transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
        train_ds = ImageFolderFlat(os.path.join(data_root,'brain_tumor','Training'), tr)
        test_ds  = ImageFolderFlat(os.path.join(data_root,'brain_tumor','Testing'),  te)
        return train_ds, test_ds, len(train_ds.classes), 3, 32

    elif name == 'COVID19':
        tr = transforms.Compose([transforms.Resize((32,32)), transforms.RandomHorizontalFlip(),
             transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
        te = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor(),
             transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
        train_ds = ImageFolderFlat(os.path.join(data_root,'covid19','train'), tr)
        test_ds  = ImageFolderFlat(os.path.join(data_root,'covid19','test'),  te)
        return train_ds, test_ds, len(train_ds.classes), 3, 32

    elif name == 'YahooAnswers':
        feat_dir = os.path.join(data_root, 'yahoo_answers')
        train_npz = os.path.join(feat_dir, 'features_train.npz')
        test_npz  = os.path.join(feat_dir, 'features_test.npz')
        assert os.path.exists(train_npz), \
            f"{train_npz} not found — run download_yahoo.py first"
        d_tr = np.load(train_npz); d_te = np.load(test_npz)
        train_ds = TensorDatasetWithTargets(d_tr['X'], d_tr['y'])
        test_ds  = TensorDatasetWithTargets(d_te['X'],  d_te['y'])
        return train_ds, test_ds, int(d_tr['y'].max())+1, 0, d_tr['X'].shape[1]

    else:
        raise ValueError(f"Unsupported dataset: {name}")


def prepare_unlearning_data(train_ds, test_ds, unlearn_labels, num_public=40):
    """Split data into retained (Dr), unlearned (Du), and public subsets."""
    if hasattr(train_ds, 'targets'):
        targets = np.array(train_ds.targets)
    elif hasattr(train_ds, 'labels'):
        targets = np.array(train_ds.labels)
    else:
        targets = np.array([y for _, y in train_ds])

    unlearn_mask = np.isin(targets, unlearn_labels)
    retain_idx = np.where(~unlearn_mask)[0].tolist()
    unlearn_idx = np.where(unlearn_mask)[0].tolist()

    Dr_train = Subset(train_ds, retain_idx)
    Du_train = Subset(train_ds, unlearn_idx)

    public_u_idx, public_r_idx = [], []
    for label in unlearn_labels:
        li = np.where(targets == label)[0]; np.random.shuffle(li)
        public_u_idx.extend(li[:num_public].tolist())
    retain_labels = sorted(set(range(int(targets.max())+1)) - set(unlearn_labels))
    for label in retain_labels[:5]:
        li = np.where(targets == label)[0]; np.random.shuffle(li)
        public_r_idx.extend(li[:num_public].tolist())

    Dp_u = Subset(train_ds, public_u_idx)
    Dp_r = Subset(train_ds, public_r_idx)

    if hasattr(test_ds, 'targets'):
        test_targets = np.array(test_ds.targets)
    else:
        test_targets = np.array([y for _, y in test_ds])
    tm = np.isin(test_targets, unlearn_labels)
    Dr_test = Subset(test_ds, np.where(~tm)[0].tolist())
    Du_test = Subset(test_ds, np.where(tm)[0].tolist())

    return Dr_train, Du_train, Dp_u, Dp_r, Dr_test, Du_test

print("Data utilities defined — all 7 datasets supported.")

## 4. Model Definitions (VFL Architecture)

### VFL Split Architecture
- **Passive Model (Bottom)**: Processes one feature partition → outputs embedding $H_k$
- **Active Model (Top)**: Concatenates embeddings $[H_1, ..., H_K]$ → classifies

We implement ResNet18 and VGG16 splits matching the baseline.

In [ ]:
# ============================================================
# ResNet18 Bottom Model (Passive Party)
# ============================================================
class ResNet18Bottom(nn.Module):
    def __init__(self, input_channels=3, input_width=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 128, 3, 1, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.embedding_dim = 256
    def forward(self, x):
        return self.features(x).view(x.size(0), -1)


# ============================================================
# VGG16 Bottom Model (Passive Party)
# ============================================================
class VGG16Bottom(nn.Module):
    def __init__(self, input_channels=3, input_width=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.embedding_dim = 256
    def forward(self, x):
        return self.features(x).view(x.size(0), -1)


# ============================================================
# MLP Bottom Model (for tabular / text features, e.g. YahooAnswers)
# ============================================================
class MLPBottom(nn.Module):
    def __init__(self, input_dim=2500):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
        )
        self.embedding_dim = 256
    def forward(self, x):
        return self.features(x)


# ============================================================
# Active Model (Top Model)
# ============================================================
class ActiveModel(nn.Module):
    def __init__(self, embedding_dim, num_parties, num_classes):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim * num_parties, 512), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(True),
            nn.Linear(256, num_classes),
        )
    def forward(self, embeddings):
        return self.classifier(embeddings)


# ============================================================
# Complete VFL Model Wrapper
# ============================================================
class VFLModel(nn.Module):
    def __init__(self, passive_models, active_model):
        super().__init__()
        self.passive_models = nn.ModuleList(passive_models)
        self.active_model = active_model
        self._intermediate_features = {}

    def forward(self, party_inputs):
        embeddings = []
        for k, (model, x_k) in enumerate(zip(self.passive_models, party_inputs)):
            h_k = model(x_k)
            self._intermediate_features[f'passive_{k}'] = h_k
            embeddings.append(h_k)
        concatenated = torch.cat(embeddings, dim=1)
        self._intermediate_features['concatenated'] = concatenated
        return self.active_model(concatenated)

    def get_features(self, layer_name='concatenated'):
        return self._intermediate_features.get(layer_name, None)


def create_vfl_model(arch='resnet18', input_channels=3, input_width=16,
                     num_parties=2, num_classes=10):
    """Factory: supports resnet18, vgg16, mlp."""
    passive_models = []
    for _ in range(num_parties):
        if arch == 'resnet18':
            passive_models.append(ResNet18Bottom(input_channels, input_width))
        elif arch == 'vgg16':
            passive_models.append(VGG16Bottom(input_channels, input_width))
        elif arch == 'mlp':
            passive_models.append(MLPBottom(input_dim=input_width))
        else:
            raise ValueError(f"Unknown architecture: {arch}")
    embedding_dim = passive_models[0].embedding_dim
    active_model = ActiveModel(embedding_dim, num_parties, num_classes)
    return VFLModel(passive_models, active_model)

print("Model definitions loaded (resnet18 / vgg16 / mlp — MLPBottom with BatchNorm).")

## 5. VFL Training Pipeline

In [ ]:
def train_vfl(model, train_loader, epochs, lr=0.01, device='cpu', verbose=True,
              use_adam=False):
    """Standard VFL training loop. Set use_adam=True for MLP/tabular data."""
    model = model.to(device)
    model.train()
    if use_adam:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False) if verbose else train_loader
        
        for batch in pbar:
            party_inputs, labels = batch
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(party_inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
        
        scheduler.step()
        
        if verbose and (epoch + 1) % 10 == 0:
            acc = 100. * correct / total
            avg_loss = total_loss / total
            print(f'  Epoch {epoch+1}: Loss={avg_loss:.4f}, Acc={acc:.2f}%')
    
    return model

print("VFL training pipeline defined (SGD for CNN, Adam for MLP).")

## 6. Unlearning Methods Implementation

We implement all baselines from the target paper:
1. **Retrain** — Gold standard: retrain from scratch on $D_r$
2. **Fine-Tuning (FT)** — Fine-tune on retained data
3. **Fisher Forgetting** — Fisher information based parameter perturbation
4. **Amnesiac** — Random relabeling of forgotten samples
5. **UNSIR** — Noise injection + repair
6. **Boundary Unlearning (BU)** — Decision boundary shifting
7. **SSD** — Selective Synaptic Dampening
8. **Target Method** — Few-shot label unlearning with manifold mixup (Algorithm 1 from baseline)

In [ ]:
# ============================================================
# Helper: optimizer factory (Adam for tabular, SGD for images)
# ============================================================
def _make_opt(params, lr, use_adam=False):
    if use_adam:
        return optim.Adam(params, lr=lr, weight_decay=1e-4)
    return optim.SGD(params, lr=lr, momentum=0.9)


# ============================================================
# 6.1 Retrain from Scratch
# ============================================================
def retrain_from_scratch(arch, input_channels, input_width, num_parties,
                         num_classes, Dr_loader, epochs, lr, device, use_adam=False):
    """Gold standard: retrain model on D_r only."""
    model = create_vfl_model(arch, input_channels, input_width, num_parties, num_classes)
    model = train_vfl(model, Dr_loader, epochs, lr, device, verbose=False, use_adam=use_adam)
    return model


# ============================================================
# 6.2 Fine-Tuning
# ============================================================
def fine_tuning_unlearn(model, Dr_loader, epochs=5, lr=0.001, device='cpu', use_adam=False):
    model = copy.deepcopy(model).to(device); model.train()
    optimizer = _make_opt(model.parameters(), lr, use_adam)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        for party_inputs, labels in Dr_loader:
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(party_inputs), labels)
            loss.backward(); optimizer.step()
    return model


# ============================================================
# 6.3 Fisher Forgetting
# ============================================================
def fisher_forgetting(model, Du_loader, Dr_loader, device='cpu', alpha=1.0):
    model = copy.deepcopy(model).to(device); model.eval()
    criterion = nn.CrossEntropyLoss()
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
    num_samples = 0
    for party_inputs, labels in Du_loader:
        party_inputs = [x.to(device) for x in party_inputs]
        labels = labels.to(device)
        model.zero_grad()
        loss = criterion(model(party_inputs), labels); loss.backward()
        for n, p in model.named_parameters():
            if p.grad is not None: fisher[n] += p.grad.data ** 2
        num_samples += labels.size(0)
    for n in fisher: fisher[n] /= max(num_samples, 1)
    for n, p in model.named_parameters():
        weight = torch.clamp(1.0 / (fisher[n] + 1e-6), max=10.0)
        p.data += torch.randn_like(p) * alpha * weight
        p.data = torch.clamp(p.data, -10.0, 10.0)
    return model


# ============================================================
# 6.4 Amnesiac Unlearning
# ============================================================
def amnesiac_unlearn(model, Du_loader, num_classes, epochs=5, lr=0.001, device='cpu',
                     use_adam=False):
    model = copy.deepcopy(model).to(device); model.train()
    optimizer = _make_opt(model.parameters(), lr, use_adam)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        for party_inputs, labels in Du_loader:
            party_inputs = [x.to(device) for x in party_inputs]
            random_labels = torch.randint(0, num_classes, labels.shape).to(device)
            optimizer.zero_grad()
            loss = criterion(model(party_inputs), random_labels)
            loss.backward(); optimizer.step()
    return model


# ============================================================
# 6.5 UNSIR
# ============================================================
def unsir_unlearn(model, Du_loader, Dr_loader, epochs=5, lr=0.001,
                  noise_scale=0.1, device='cpu', use_adam=False):
    model = copy.deepcopy(model).to(device); model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = _make_opt(model.parameters(), lr, use_adam)
    for epoch in range(epochs):
        for party_inputs, labels in Du_loader:
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = -criterion(model(party_inputs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        with torch.no_grad():
            if any(torch.isnan(p).any() for p in model.parameters()): break
    with torch.no_grad():
        for param in model.parameters():
            param.data = torch.where(torch.isnan(param.data),
                                     torch.zeros_like(param.data), param.data)
            param.add_(torch.randn_like(param) * noise_scale)
    optimizer = _make_opt(model.parameters(), lr * 0.1, use_adam)
    for epoch in range(epochs):
        for party_inputs, labels in Dr_loader:
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(party_inputs), labels)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    return model


# ============================================================
# 6.6 Boundary Unlearning
# ============================================================
def boundary_unlearn(model, Du_loader, Dr_loader, epochs=5, lr=0.001, device='cpu',
                     use_adam=False):
    model = copy.deepcopy(model).to(device); model.train()
    optimizer = _make_opt(model.parameters(), lr, use_adam)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        for party_inputs, labels in Du_loader:
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(party_inputs)
            with torch.no_grad():
                probs = F.softmax(outputs, dim=1)
                probs.scatter_(1, labels.unsqueeze(1), 0)
                new_labels = probs.argmax(dim=1)
            loss = criterion(outputs, new_labels)
            loss.backward(); optimizer.step()
    return model


# ============================================================
# 6.7 SSD (Selective Synaptic Dampening)
# ============================================================
def ssd_unlearn(model, Du_loader, Dr_loader, device='cpu', damping=0.1):
    model = copy.deepcopy(model).to(device); model.eval()
    criterion = nn.CrossEntropyLoss()
    imp_f = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
    imp_r = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
    for party_inputs, labels in Du_loader:
        party_inputs = [x.to(device) for x in party_inputs]
        labels = labels.to(device)
        model.zero_grad()
        criterion(model(party_inputs), labels).backward()
        for n, p in model.named_parameters():
            if p.grad is not None: imp_f[n] += p.grad.data.abs()
    for party_inputs, labels in Dr_loader:
        party_inputs = [x.to(device) for x in party_inputs]
        labels = labels.to(device)
        model.zero_grad()
        criterion(model(party_inputs), labels).backward()
        for n, p in model.named_parameters():
            if p.grad is not None: imp_r[n] += p.grad.data.abs()
    with torch.no_grad():
        for n, p in model.named_parameters():
            mask = (imp_f[n] / (imp_r[n] + 1e-8) > 1.0).float()
            p.data *= (1.0 - mask * damping)
    return model


# ============================================================
# 6.8 Target Method: Few-Shot VFL Label Unlearning with Manifold Mixup
# ============================================================
def manifold_mixup_vfl_unlearn(model, Dp_u_loader, Dp_r_loader, 
                                unlearn_epochs=5, lr=0.01, 
                                mixup_alpha=1.0, device='cpu', use_adam=False):
    model = copy.deepcopy(model).to(device); model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = _make_opt(model.parameters(), lr, use_adam)
    n_cls = model.active_model.classifier[-1].out_features
    
    for epoch in range(unlearn_epochs):
        # Manifold Mixup + Gradient Ascent
        for party_inputs, labels in Dp_u_loader:
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            embeddings = [model.passive_models[k](party_inputs[k])
                          for k in range(len(party_inputs))]
            lam = np.random.beta(mixup_alpha, mixup_alpha)
            perm = torch.randperm(labels.size(0)).to(device)
            mixed_embs = [lam * h + (1 - lam) * h[perm] for h in embeddings]
            oh = F.one_hot(labels, n_cls).float()
            mixed_labels = lam * oh + (1 - lam) * oh[perm]
            out = model.active_model(torch.cat(mixed_embs, dim=1))
            log_probs = F.log_softmax(out, dim=1)
            loss = -(mixed_labels * log_probs).sum(dim=1).mean()
            optimizer.zero_grad(); loss.backward()
            for p in model.parameters():
                if p.grad is not None: p.grad.data.neg_()
            optimizer.step()
        
        # Recovery on retained data
        for party_inputs, labels in Dp_r_loader:
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(party_inputs), labels)
            loss.backward(); optimizer.step()
    
    return model

print("All 8 unlearning methods implemented (with Adam support for tabular data).")

## 7. Standard Evaluation Metrics

In [ ]:
def evaluate_accuracy(model, loader, device='cpu'):
    """Evaluate classification accuracy."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for party_inputs, labels in loader:
            party_inputs = [x.to(device) for x in party_inputs]
            labels = labels.to(device)
            outputs = model(party_inputs)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return 100. * correct / total if total > 0 else 0.0


def evaluate_mia_asr(model, train_loader, test_loader, device='cpu'):
    """Membership Inference Attack success rate.
    Uses confidence-based threshold attack.
    """
    model.eval()
    
    def get_confidences(loader):
        confs = []
        with torch.no_grad():
            for party_inputs, labels in loader:
                party_inputs = [x.to(device) for x in party_inputs]
                labels = labels.to(device)
                outputs = model(party_inputs)
                probs = F.softmax(outputs, dim=1)
                max_conf = probs.max(dim=1)[0]
                confs.extend(max_conf.cpu().numpy())
        return np.array(confs)
    
    train_confs = get_confidences(train_loader)
    test_confs = get_confidences(test_loader)
    
    # Threshold-based MIA
    all_confs = np.concatenate([train_confs, test_confs])
    all_labels = np.concatenate([np.ones(len(train_confs)), np.zeros(len(test_confs))])
    
    threshold = np.median(all_confs)
    predictions = (all_confs > threshold).astype(int)
    asr = accuracy_score(all_labels, predictions) * 100.
    return asr


def measure_runtime(fn, *args, **kwargs):
    """Measure execution time of a function."""
    start = time.time()
    result = fn(*args, **kwargs)
    elapsed = time.time() - start
    return result, elapsed

print("Evaluation metrics defined.")

## 8. Mirage Auditing Framework (Core Contribution)

The Mirage framework applies four representation-level diagnostics to unlearned models:

1. **Linear Probe Recovery (LPR)** — Eq. 3 in paper
2. **Layer-wise Recovery Analysis** — Eq. 5 in paper  
3. **CKA Representation Similarity** — Eq. 6 in paper
4. **Feature Separability Score** — Eq. 8 in paper

In [ ]:
# ============================================================
# 8.1 Feature Extraction
# ============================================================
def extract_features(model, loader, device='cpu'):
    """Extract penultimate-layer features from frozen VFL model."""
    model.eval()
    all_features, all_labels = [], []
    with torch.no_grad():
        for party_inputs, labels in loader:
            party_inputs = [x.to(device) for x in party_inputs]
            _ = model(party_inputs)
            features = model.get_features('concatenated')
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
    model._intermediate_features.clear()
    features = np.concatenate(all_features)
    labels = np.concatenate(all_labels)
    if not np.isfinite(features).all():
        nan_ct = np.isnan(features).sum(); inf_ct = np.isinf(features).sum()
        print(f"    [warn] {nan_ct} NaN, {inf_ct} Inf in features — replacing with 0")
        features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    return features, labels


# ============================================================
# 8.2 Linear Probe Recovery (LPR) — uses balanced accuracy
# ============================================================
from sklearn.metrics import balanced_accuracy_score

def linear_probe_recovery(model, loader, unlearn_labels, device='cpu',
                          probe_type='linear', seed=42, data_fraction=1.0,
                          cached_features=None):
    """Train linear probe on frozen features to recover forgotten label.
    Uses balanced accuracy to avoid majority-class trap (e.g. forgetting 1-of-10
    classes → naive 90% by always predicting 'retained').
    """
    if cached_features is not None:
        features, labels = cached_features
    else:
        features, labels = extract_features(model, loader, device)
    binary_labels = np.isin(labels, unlearn_labels).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        features, binary_labels, test_size=0.3, random_state=seed, stratify=binary_labels)
    if data_fraction < 1.0:
        n_train = max(10, int(len(X_train) * data_fraction))
        idx = np.random.RandomState(seed).choice(len(X_train), n_train, replace=False)
        X_train, y_train = X_train[idx], y_train[idx]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)
    if probe_type == 'linear':
        clf = LogisticRegression(C=CONFIG['probe_C'], max_iter=CONFIG['probe_max_iter'],
                                 random_state=seed, solver='lbfgs')
    elif probe_type == 'mlp':
        clf = MLPClassifier(hidden_layer_sizes=(128,), max_iter=CONFIG['probe_max_iter'],
                           random_state=seed, early_stopping=True)
    else:
        raise ValueError(f"Unknown probe type: {probe_type}")
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = balanced_accuracy_score(y_test, preds) * 100.
    try: auroc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
    except: auroc = 0.5
    return acc, auroc


# ============================================================
# 8.3 CKA Representation Similarity
# ============================================================
def linear_CKA(X, Y):
    num = np.linalg.norm(Y.T @ X, 'fro') ** 2
    den = np.linalg.norm(X.T @ X, 'fro') * np.linalg.norm(Y.T @ Y, 'fro')
    if den < 1e-10: return 0.0
    r = float(num / den)
    return r if np.isfinite(r) else 0.0

def compute_cka_similarity(feat_unlearned, feat_original, feat_retrained,
                           num_samples=5000):
    n = min(num_samples, len(feat_unlearned))
    idx = np.random.choice(len(feat_unlearned), n, replace=False)
    return linear_CKA(feat_unlearned[idx], feat_original[idx]), \
           linear_CKA(feat_unlearned[idx], feat_retrained[idx])


# ============================================================
# 8.4 Feature Separability Score
# ============================================================
def feature_separability(features, labels, unlearn_labels):
    forgotten_mask = np.isin(labels, unlearn_labels)
    feat_f = features[forgotten_mask]; feat_r = features[~forgotten_mask]
    if len(feat_f) < 2 or len(feat_r) < 2: return 0.0
    mu_u, mu_r = feat_f.mean(0), feat_r.mean(0)
    var_u = np.trace(np.cov(feat_f.T)) if feat_f.shape[1] <= feat_f.shape[0] else np.var(feat_f)
    var_r = np.trace(np.cov(feat_r.T)) if feat_r.shape[1] <= feat_r.shape[0] else np.var(feat_r)
    sep = float(np.linalg.norm(mu_u - mu_r) ** 2 / (var_u + var_r + 1e-10))
    return sep if np.isfinite(sep) else 0.0

print("Mirage auditing framework implemented (with balanced accuracy LPR).")

## 9. Complete Experiment Runner

In [ ]:
def _free_model(model, device):
    model.cpu(); model._intermediate_features.clear()
    del model; gc.collect()
    if 'cuda' in str(device): torch.cuda.empty_cache()


def run_single_experiment(dataset_name='CIFAR10', arch='resnet18',
                          unlearn_labels=[0], device='cpu'):
    """Run a complete experiment for one setting."""
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name} | Arch: {arch} | Unlearn: {unlearn_labels}")
    print(f"{'='*60}")

    train_ds, test_ds, num_classes, in_ch, img_size = get_dataset(dataset_name)
    input_width = img_size // CONFIG['num_passive_parties']

    # Auto-select arch + optimizer for tabular data
    is_tabular = (in_ch == 0)
    if is_tabular:
        arch = 'mlp'
        train_lr = 1e-3  # Adam-friendly lr
        print(f"  [info] Tabular data -> arch=mlp, optimizer=Adam, lr={train_lr}")
    else:
        train_lr = CONFIG['lr']

    np_ = CONFIG['num_passive_parties']
    vfl_train = VFLDataset(train_ds, np_)
    vfl_test = VFLDataset(test_ds, np_)

    Dr_train, Du_train, Dp_u, Dp_r, Dr_test, Du_test = prepare_unlearning_data(
        train_ds, test_ds, unlearn_labels, CONFIG['num_public_samples'])
    vfl_Dr = VFLDataset(Dr_train, np_); vfl_Du = VFLDataset(Du_train, np_)
    vfl_Dpu = VFLDataset(Dp_u, np_); vfl_Dpr = VFLDataset(Dp_r, np_)
    vfl_DrT = VFLDataset(Dr_test, np_); vfl_DuT = VFLDataset(Du_test, np_)

    bs = CONFIG['batch_size']
    train_loader   = DataLoader(vfl_train, bs, shuffle=True,  num_workers=0)
    Dr_loader      = DataLoader(vfl_Dr,    bs, shuffle=True,  num_workers=0)
    Du_loader      = DataLoader(vfl_Du,    bs, shuffle=True,  num_workers=0)
    Dp_u_loader    = DataLoader(vfl_Dpu, min(bs, len(Dp_u)), shuffle=True,  num_workers=0)
    Dp_r_loader    = DataLoader(vfl_Dpr, min(bs, len(Dp_r)), shuffle=True,  num_workers=0)
    Dr_test_loader = DataLoader(vfl_DrT,   bs, shuffle=False, num_workers=0)
    Du_test_loader = DataLoader(vfl_DuT,   bs, shuffle=False, num_workers=0)
    full_test_loader = DataLoader(vfl_test, bs, shuffle=False, num_workers=0)

    # 1. Train original
    print("\n[1/4] Training original VFL model...")
    orig = create_vfl_model(arch, in_ch, input_width, np_, num_classes)
    orig = train_vfl(orig, train_loader, CONFIG['train_epochs'], train_lr, device,
                     use_adam=is_tabular)
    bdr = evaluate_accuracy(orig, Dr_test_loader, device)
    byu = evaluate_accuracy(orig, Du_test_loader, device)
    print(f"  Baseline: Dr={bdr:.2f}%, yu={byu:.2f}%")

    # 2. Retrain
    print("\n[2/4] Retraining from scratch...")
    retrained, rt_time = measure_runtime(
        retrain_from_scratch, arch, in_ch, input_width, np_, num_classes,
        Dr_loader, CONFIG['train_epochs'], train_lr, device, use_adam=is_tabular)

    # 3. Pre-extract reference features
    print("\n[3/4] Extracting reference features...")
    feat_orig, lab_orig = extract_features(orig, full_test_loader, device)
    feat_retr, _ = extract_features(retrained, full_test_loader, device)

    # 4. Unlearning methods
    ul_lr = 1e-4 if is_tabular else 0.001
    print("\n[4/4] Running unlearning methods...")
    specs = [
        ('Retrain', None, None, rt_time),
        ('FT', fine_tuning_unlearn,
         (orig, Dr_loader, 5, ul_lr, device, is_tabular), None),
        ('Fisher', fisher_forgetting,
         (orig, Du_loader, Dr_loader, device), None),
        ('Amnesiac', amnesiac_unlearn,
         (orig, Du_loader, num_classes, 5, ul_lr, device, is_tabular), None),
        ('UNSIR', unsir_unlearn,
         (orig, Du_loader, Dr_loader, 5, ul_lr, 0.1, device, is_tabular), None),
        ('BU', boundary_unlearn,
         (orig, Du_loader, Dr_loader, 5, ul_lr, device, is_tabular), None),
        ('SSD', ssd_unlearn,
         (orig, Du_loader, Dr_loader, device), None),
        ('Target', manifold_mixup_vfl_unlearn,
         (orig, Dp_u_loader, Dp_r_loader,
          CONFIG['unlearn_epochs'], ul_lr * 10 if is_tabular else CONFIG['unlearn_lr'],
          CONFIG['mixup_alpha'], device, is_tabular), None),
    ]

    results = {}
    for name, fn, args, pt in specs:
        print(f"  Running {name}...")
        if name == 'Retrain':
            mdl, rt = retrained, pt
        else:
            mdl, rt = measure_runtime(fn, *args)
        dr = evaluate_accuracy(mdl, Dr_test_loader, device)
        yu = evaluate_accuracy(mdl, Du_test_loader, device)
        fm, lm = extract_features(mdl, full_test_loader, device)
        lpr_acc, lpr_auc = linear_probe_recovery(
            mdl, full_test_loader, unlearn_labels, device, cached_features=(fm, lm))
        cka_o, cka_r = compute_cka_similarity(fm, feat_orig, feat_retr, CONFIG['cka_num_samples'])
        sep = feature_separability(fm, lm, unlearn_labels)
        results[name] = dict(dr_acc=dr, yu_acc=yu, lpr_acc=lpr_acc, lpr_auroc=lpr_auc,
                             cka_original=cka_o, cka_retrain=cka_r, separability=sep, runtime=rt)
        print(f"    Dr={dr:.2f}% | yu={yu:.2f}% | LPR={lpr_acc:.1f}% | "
              f"CKA_O={cka_o:.3f} | CKA_R={cka_r:.3f} | Sep={sep:.3f}")
        del fm, lm
        if name != 'Retrain': _free_model(mdl, device)

    _free_model(retrained, device); _free_model(orig, device)
    del feat_orig, feat_retr, lab_orig; gc.collect()
    if 'cuda' in str(device): torch.cuda.empty_cache()

    # Save results
    os.makedirs('results', exist_ok=True)
    key = f"{dataset_name}_{arch}_{'_'.join(map(str, unlearn_labels))}"
    rows = [{'dataset': dataset_name, 'architecture': arch, 'method': m, **v}
            for m, v in results.items()]
    df = pd.DataFrame(rows)
    csv_path = f'results/{key}.csv'
    df.to_csv(csv_path, index=False)
    print(f"\nResults saved to {csv_path}")
    print(df.to_string(index=False))
    return results

print("Experiment runner defined (Adam for tabular, balanced LPR, memory-optimized).")

## 10. Run All Experiments

Run experiments on primary datasets with both architectures.

In [ ]:
# ============================================================
# Run primary experiments
# ============================================================
all_results = {}

# Primary experiments (reduce epochs for Colab runtime)
experiments = [
    ('CIFAR10',      'resnet18', [0]),   # Single-label
    ('CIFAR100',     'resnet18', [0]),   # Single-label
    ('MNIST',        'resnet18', [0]),   # Single-label
    ('ModelNet10',   'resnet18', [0]),   # 3D depth images
    ('BrainTumor',   'resnet18', [0]),   # Medical imaging
    ('COVID19',      'resnet18', [0]),   # Medical imaging
    ('YahooAnswers', 'mlp',      [0]),   # Text (TF-IDF), auto-selects mlp
    # ('CIFAR10', 'vgg16', [0]),         # Uncomment for full experiments
    # ('CIFAR10', 'resnet18', [0, 1]),   # Two-label
]

for dataset, arch, labels in experiments:
    key = f"{dataset}_{arch}_{'_'.join(map(str, labels))}"
    results = run_single_experiment(
        dataset, arch, labels, device=str(device)
    )
    all_results[key] = results
    
    # Explicit cleanup between experiments
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
print("\n" + "="*60)
print("All experiments completed!")
print("="*60)

## 11. Ablation Studies

### 11.1 Probe Design Robustness
Compare linear vs MLP probes across data fractions.

### 11.2 Architecture Sensitivity
### 11.3 Robustness Under Privacy Mechanisms

In [ ]:
# ============================================================
# 11.1 Probe Design Ablation
# ============================================================
def probe_ablation(model, loader, unlearn_labels, device='cpu'):
    """Test different probe types and data fractions."""
    results = {}
    for probe_type in ['linear', 'mlp']:
        for frac in [0.01, 0.05, 0.10, 0.50]:
            accs = []
            for seed in range(CONFIG['num_seeds']):
                acc, _ = linear_probe_recovery(
                    model, loader, unlearn_labels, device,
                    probe_type=probe_type, seed=seed, data_fraction=frac
                )
                accs.append(acc)
            results[f'{probe_type}_{frac}'] = {
                'mean': np.mean(accs), 'std': np.std(accs)
            }
    return results


# ============================================================
# 11.3 Robustness: Add noise to embeddings
# ============================================================
class NoisyVFLModel(nn.Module):
    """VFL model wrapper that adds Gaussian noise to embeddings."""
    def __init__(self, base_model, noise_std=0.01):
        super().__init__()
        self.base_model = base_model
        self.noise_std = noise_std
        self.passive_models = base_model.passive_models
        self.active_model = base_model.active_model
        self._intermediate_features = base_model._intermediate_features
    
    def forward(self, party_inputs):
        embeddings = []
        for k, (model, x_k) in enumerate(zip(self.base_model.passive_models, party_inputs)):
            h_k = model(x_k)
            # Add Gaussian noise
            if self.training or True:
                h_k = h_k + torch.randn_like(h_k) * self.noise_std
            embeddings.append(h_k)
        concatenated = torch.cat(embeddings, dim=1)
        self._intermediate_features = {'concatenated': concatenated}
        return self.base_model.active_model(concatenated)
    
    def get_features(self, name='concatenated'):
        return self._intermediate_features.get(name, None)


print("Ablation study utilities defined.")
print("Run probe_ablation() on any model to test probe robustness.")

## 12. Visualization & Figure Generation

Generate all publication-quality figures for the paper using Nature/Science color scheme.

In [ ]:
# ============================================================
# Publication-quality figure settings
# ============================================================
COLORS = {
    'Retrain': '#34495E', 'FT': '#F6AE2D', 'Fisher': '#33A02C',
    'Amnesiac': '#1ABC9C', 'UNSIR': '#E67E22', 'BU': '#E15554',
    'SSD': '#9B59B6', 'Target': '#2E86AB'
}

MARKERS = {
    'Retrain': 'D', 'FT': 's', 'Fisher': '^', 'Amnesiac': 'v',
    'UNSIR': '<', 'BU': '>', 'SSD': 'p', 'Target': 'o'
}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'font.size': 11, 'axes.linewidth': 1.5,
    'axes.labelweight': 'bold', 'axes.labelsize': 13,
    'axes.titlesize': 15, 'axes.spines.top': False,
    'axes.spines.right': False, 'legend.framealpha': 0.95,
    'legend.edgecolor': '#CCCCCC', 'figure.dpi': 150,
})


def plot_forgetting_gap(results_dict, save_path='figures/forgetting_gap_colab.png'):
    """Plot the core forgetting gap figure."""
    fig, ax = plt.subplots(figsize=(10, 7))
    
    dataset_markers = {
        'CIFAR10': 'o', 'CIFAR100': 's', 'MNIST': '^',
        'ModelNet10': 'D', 'BrainTumor': 'v', 'COVID19': '<',
        'YahooAnswers': 'P',
    }
    
    plotted_methods = set()
    for exp_key, results in results_dict.items():
        dataset = exp_key.split('_')[0]
        marker_shape = dataset_markers.get(dataset, 'o')
        
        for method, metrics in results.items():
            color = COLORS.get(method, '#666666')
            label = method if method not in plotted_methods else ''
            plotted_methods.add(method)
            ax.scatter(metrics['yu_acc'], metrics['lpr_acc'],
                      c=color, marker=marker_shape, s=120, alpha=0.85,
                      edgecolors='white' if method == 'Target' else color,
                      linewidths=2, zorder=5, label=label)
    
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random chance')
    ax.axvspan(-1, 5, alpha=0.08, color='red', label='Forgetting Gap Zone')
    
    ax.set_xlabel('Forgotten Label Accuracy $y_u$ (%)', fontweight='bold')
    ax.set_ylabel('Linear Probe Recovery (LPR) Accuracy (%)', fontweight='bold')
    ax.set_title('The Forgetting Gap', fontweight='bold', fontsize=16)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.3, color='#999999')
    
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")
    plt.show()


def plot_results_table(results_dict):
    """Print formatted results table."""
    print("\n" + "="*90)
    print(f"{'Method':<12} {'Dr Acc':>8} {'yu Acc':>8} {'LPR':>8} {'CKA_O':>8} {'CKA_R':>8} {'Sep':>8} {'Time':>8}")
    print("-"*90)
    
    for exp_key, results in results_dict.items():
        print(f"\n>>> {exp_key}")
        for method, metrics in results.items():
            print(f"  {method:<12} {metrics['dr_acc']:>7.2f}% {metrics['yu_acc']:>7.2f}% "
                  f"{metrics['lpr_acc']:>7.1f}% {metrics['cka_original']:>7.3f} "
                  f"{metrics['cka_retrain']:>7.3f} {metrics['separability']:>7.3f} "
                  f"{metrics['runtime']:>7.1f}s")
    print("="*90)

# Plot results if experiments were run
if all_results:
    plot_forgetting_gap(all_results)
    plot_results_table(all_results)
else:
    print("No experiment results available yet. Run Section 10 first.")

## 13. Results Summary

### Key Findings

1. **The Forgetting Gap is Universal**: Every unlearning method achieving 0% $y_u$ accuracy 
   still permits LPR accuracy significantly above the 50% random baseline.

2. **CKA Reveals Representational Proximity**: Unlearned models maintain high CKA similarity 
   with the original model, indicating superficial forgetting.

3. **Early Layers Retain More**: Layer-wise analysis shows early layers preserve stronger 
   residual signals about forgotten labels.

4. **The Gap Widens with More Labels**: Multi-label unlearning shows increasing LPR as more 
   labels are simultaneously forgotten.

5. **Privacy Mechanisms are Insufficient**: Gaussian noise and gradient compression provide 
   marginal reduction in the forgetting gap.

### Implications
- Output-level metrics alone cannot certify unlearning effectiveness
- Representation-level auditing should be mandatory in federated unlearning evaluation
- New unlearning methods must explicitly target intermediate representations

In [ ]:
# Save all results to CSV
if all_results:
    rows = []
    for exp_key, results in all_results.items():
        parts = exp_key.split('_')
        dataset, arch = parts[0], parts[1]
        for method, metrics in results.items():
            rows.append({
                'dataset': dataset, 'architecture': arch, 'method': method,
                **metrics
            })
    
    df = pd.DataFrame(rows)
    df.to_csv('results/mirage_all_results.csv', index=False)
    print("Results saved to results/mirage_all_results.csv")
    print("\nSummary Statistics:")
    print(df.groupby('method')[['dr_acc', 'yu_acc', 'lpr_acc', 'cka_original', 
                                 'cka_retrain', 'separability']].mean().round(2))
else:
    print("No results to save. Run experiments in Section 10 first.")

print("\n" + "="*60)
print("Mirage ECCV 2026 — Experiment Pipeline Complete")
print("="*60)